# Second Order Ordinary Differential Equations
# Équations Différentielles du Second Ordre

**AIMS Master's Programme — ODE Course**

We study second-order linear ODEs with constant coefficients, focusing on the damped harmonic oscillator (oscillateur amorti). We cover the characteristic equation, method of undetermined coefficients (méthode des coefficients indéterminés), variation of parameters (variation des constantes), and spring-mass system visualisation.

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

plt.rcParams.update({'figure.figsize': (8, 5), 'font.size': 12})

## 1. Homogeneous Equations with Constant Coefficients

Consider the general form:
$$a y'' + b y' + c y = 0$$

We seek solutions of the form $y = e^{\lambda t}$, leading to the **characteristic equation** (équation caractéristique):
$$a\lambda^2 + b\lambda + c = 0$$

The discriminant $\Delta = b^2 - 4ac$ determines the nature of the solution:

| $\Delta$ | Roots | Solution type | Physical behaviour |
|:---:|:---:|:---|:---|
| $> 0$ | Two distinct real $\lambda_1, \lambda_2$ | $y = C_1 e^{\lambda_1 t} + C_2 e^{\lambda_2 t}$ | **Overdamped** (suramortissement) |
| $= 0$ | Repeated real $\lambda$ | $y = (C_1 + C_2 t)e^{\lambda t}$ | **Critically damped** (amortissement critique) |
| $< 0$ | Complex $\alpha \pm i\beta$ | $y = e^{\alpha t}(C_1\cos\beta t + C_2\sin\beta t)$ | **Underdamped** (sous-amortissement) |

### The Damped Oscillator: $m y'' + c y' + k y = 0$

In [ ]:
# Damped harmonic oscillator: my'' + cy' + ky = 0
# Rewrite as system: y' = v, v' = -(c/m)v - (k/m)y

def damped_oscillator(t, state, m, c, k):
    y, v = state
    return [v, -(c/m)*v - (k/m)*y]

m, k = 1.0, 4.0  # mass and spring constant
y0, v0 = 1.0, 0.0  # initial displacement and velocity
t_span = (0, 10)
t_eval = np.linspace(*t_span, 500)

# Three damping regimes
# Critical damping: c_cr = 2*sqrt(m*k)
c_cr = 2 * np.sqrt(m * k)
damping_cases = {
    f'Underdamped (c={0.5*c_cr:.1f})': 0.5 * c_cr,
    f'Critically damped (c={c_cr:.1f})': c_cr,
    f'Overdamped (c={2*c_cr:.1f})': 2 * c_cr,
}

plt.figure(figsize=(10, 6))
colors = ['blue', 'red', 'green']
for (label, c_val), color in zip(damping_cases.items(), colors):
    sol = solve_ivp(damped_oscillator, t_span, [y0, v0],
                    args=(m, c_val, k), t_eval=t_eval)
    plt.plot(sol.t, sol.y[0], color=color, linewidth=2, label=label)

plt.axhline(y=0, color='gray', linestyle=':', alpha=0.5)
plt.xlabel('Time $t$')
plt.ylabel('Displacement $y(t)$')
plt.title('Damped Harmonic Oscillator: Three Regimes')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
# Figure: Comparison of underdamped (oscillatory decay), critically damped
# (fastest non-oscillatory decay), and overdamped (slow exponential decay).

## 2. Characteristic Equation Analysis

Let us write code that, given $a, b, c$, classifies the solution type and computes the roots.

In [ ]:
def classify_ode(a, b, c):
    """Classify ay'' + by' + cy = 0 via characteristic equation."""
    discriminant = b**2 - 4*a*c
    print(f"Characteristic equation: {a}λ² + {b}λ + {c} = 0")
    print(f"Discriminant Δ = {discriminant:.4f}")
    
    if discriminant > 0:
        l1 = (-b + np.sqrt(discriminant)) / (2*a)
        l2 = (-b - np.sqrt(discriminant)) / (2*a)
        print(f"Two real roots: λ₁ = {l1:.4f}, λ₂ = {l2:.4f}")
        print("→ Overdamped (suramortissement)")
        print(f"  y(t) = C₁·exp({l1:.4f}·t) + C₂·exp({l2:.4f}·t)")
    elif discriminant == 0:
        l = -b / (2*a)
        print(f"Repeated root: λ = {l:.4f}")
        print("→ Critically damped (amortissement critique)")
        print(f"  y(t) = (C₁ + C₂·t)·exp({l:.4f}·t)")
    else:
        alpha = -b / (2*a)
        beta = np.sqrt(-discriminant) / (2*a)
        print(f"Complex roots: λ = {alpha:.4f} ± {beta:.4f}i")
        print("→ Underdamped (sous-amortissement)")
        print(f"  y(t) = exp({alpha:.4f}·t)·[C₁·cos({beta:.4f}·t) + C₂·sin({beta:.4f}·t)]")

print("=== Example 1: Underdamped ===")
classify_ode(1, 1, 4)
print("\n=== Example 2: Overdamped ===")
classify_ode(1, 5, 4)
print("\n=== Example 3: Critically damped ===")
classify_ode(1, 4, 4)

## 3. Non-Homogeneous Equations: Method of Undetermined Coefficients

For $ay'' + by' + cy = g(t)$, the general solution is $y = y_h + y_p$ where $y_h$ is the homogeneous solution and $y_p$ is a **particular solution** (solution particulière).

When $g(t)$ is a polynomial, exponential, sine/cosine, or combination thereof, we can *guess* the form of $y_p$.

### Example: $y'' + 4y = \cos(t)$

- Homogeneous: $y_h = C_1\cos(2t) + C_2\sin(2t)$
- Guess: $y_p = A\cos(t) + B\sin(t)$
- Substituting: $-A\cos(t) - B\sin(t) + 4A\cos(t) + 4B\sin(t) = \cos(t)$
- So $3A = 1, 3B = 0$, giving $y_p = \frac{1}{3}\cos(t)$.

In [ ]:
# y'' + 4y = cos(t),  y(0) = 0, y'(0) = 0
def forced_oscillator(t, state):
    y, v = state
    return [v, np.cos(t) - 4*y]

t_span = (0, 20)
t_eval = np.linspace(*t_span, 500)
sol_nh = solve_ivp(forced_oscillator, t_span, [0, 0], t_eval=t_eval)

# Exact: y(0)=0, y'(0)=0 => y = (1/3)cos(t) - (1/3)cos(2t)
y_exact_nh = (1/3)*np.cos(t_eval) - (1/3)*np.cos(2*t_eval)

plt.figure()
plt.plot(t_eval, y_exact_nh, 'b-', label='Exact', linewidth=2)
plt.plot(sol_nh.t, sol_nh.y[0], 'r--', label='Numerical', linewidth=2)
plt.xlabel('$t$'); plt.ylabel('$y(t)$')
plt.title('$y\\prime\\prime + 4y = \\cos(t)$ — Undetermined Coefficients')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
# Figure: Beat phenomenon from superposition of two close frequencies.

## 4. Variation of Parameters (Variation des constantes)

For any non-homogeneous linear ODE $y'' + p(t)y' + q(t)y = g(t)$ — even when $p, q$ are not constant — we can find $y_p$ using **variation of parameters**.

If $y_1, y_2$ form a fundamental set for the homogeneous equation, then:
$$y_p = -y_1 \int \frac{y_2\,g}{W}\,dt + y_2 \int \frac{y_1\,g}{W}\,dt$$
where $W = y_1 y_2' - y_2 y_1'$ is the **Wronskian** (wronskien).

### Example: $y'' + y = \sec(t)$

Here $y_1 = \cos t$, $y_2 = \sin t$, $W = 1$, and $g(t) = \sec(t)$.

$$y_p = -\cos t \int \sin t \sec t\,dt + \sin t \int \cos t \sec t\,dt = -\cos t \int \tan t\,dt + \sin t \int dt$$
$$= \cos t \ln|\cos t| + t\sin t$$

In [ ]:
# y'' + y = sec(t), verified numerically on (-pi/2, pi/2)
def vop_system(t, state):
    y, v = state
    return [v, 1/np.cos(t) - y]

t_span_vop = (0, 1.4)  # stay away from pi/2 ≈ 1.5708
t_eval_vop = np.linspace(0, 1.4, 300)
sol_vop = solve_ivp(vop_system, t_span_vop, [0, 0], t_eval=t_eval_vop, max_step=0.01)

# Exact particular solution: cos(t)*ln|cos(t)| + t*sin(t)
y_exact_vop = np.cos(t_eval_vop) * np.log(np.abs(np.cos(t_eval_vop))) + t_eval_vop * np.sin(t_eval_vop)

plt.figure()
plt.plot(t_eval_vop, y_exact_vop, 'b-', label='Exact (variation of parameters)', linewidth=2)
plt.plot(sol_vop.t, sol_vop.y[0], 'r--', label='Numerical', linewidth=2)
plt.xlabel('$t$'); plt.ylabel('$y(t)$')
plt.title('$y\\prime\\prime + y = \\sec(t)$ — Variation of Parameters')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
# Figure: Exact solution from variation of parameters matches numerical integration.

## 5. Spring-Mass System Visualisation

We animate a mass-spring system (système masse-ressort) to build physical intuition. The underdamped case shows oscillatory decay.

In [ ]:
# Animated spring-mass system (underdamped)
m_anim, c_anim, k_anim = 1.0, 0.3, 4.0
t_anim = np.linspace(0, 15, 300)
sol_anim = solve_ivp(damped_oscillator, (0, 15), [1.5, 0],
                     args=(m_anim, c_anim, k_anim), t_eval=t_anim)

fig_anim, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5),
                                     gridspec_kw={'width_ratios': [1, 2]})

# Left panel: spring-mass schematic
ax1.set_xlim(-1, 1)
ax1.set_ylim(-2.5, 2.5)
ax1.set_aspect('equal')
ax1.set_title('Spring-Mass')
ax1.axhline(y=0, color='gray', linestyle='--', alpha=0.5)

mass_patch = plt.Rectangle((-0.3, 0), 0.6, 0.4, fc='steelblue', ec='black')
ax1.add_patch(mass_patch)
spring_line, = ax1.plot([], [], 'k-', linewidth=2)

# Right panel: displacement vs time
ax2.set_xlim(0, 15)
ax2.set_ylim(-2, 2)
ax2.set_xlabel('Time $t$')
ax2.set_ylabel('Displacement $y(t)$')
ax2.set_title('Underdamped Oscillation')
ax2.grid(True, alpha=0.3)
line_y, = ax2.plot([], [], 'b-', linewidth=2)
dot_y, = ax2.plot([], [], 'ro', markersize=8)

def init():
    line_y.set_data([], [])
    dot_y.set_data([], [])
    return line_y, dot_y, mass_patch

def animate(i):
    y_pos = sol_anim.y[0][i]
    # Update mass position
    mass_patch.set_y(y_pos - 0.2)
    # Simple spring visualisation
    n_coils = 10
    spring_y = np.linspace(2, y_pos, n_coils * 4)
    spring_x = 0.2 * np.sin(np.linspace(0, n_coils * 2 * np.pi, len(spring_y)))
    spring_line.set_data(spring_x, spring_y)
    # Update trajectory
    line_y.set_data(sol_anim.t[:i+1], sol_anim.y[0][:i+1])
    dot_y.set_data([sol_anim.t[i]], [sol_anim.y[0][i]])
    return line_y, dot_y, mass_patch, spring_line

anim = FuncAnimation(fig_anim, animate, init_func=init,
                     frames=len(t_anim), interval=40, blit=True)
plt.tight_layout()
HTML(anim.to_jshtml())
# Figure: Animation of underdamped spring-mass system with displacement trace.

## 6. Exercise: Forced Oscillations and Resonance (Résonance)

Consider the forced, damped oscillator:
$$y'' + 2\zeta\omega_0 y' + \omega_0^2 y = F_0 \cos(\omega t)$$

where $\omega_0 = \sqrt{k/m}$ is the natural frequency (fréquence propre), $\zeta = c/(2m\omega_0)$ is the damping ratio, and $\omega$ is the forcing frequency (fréquence de forçage).

**Resonance** occurs when the forcing frequency $\omega$ is near the natural frequency $\omega_0$. The amplitude of the steady-state response is maximized.

**Tasks:**
1. Fix $\omega_0 = 2$, $\zeta = 0.1$, $F_0 = 1$. Solve for $\omega \in \{1.0, 1.9, 2.0, 2.1, 3.0\}$.
2. Plot each solution and observe the amplitude growth near resonance.
3. Plot the **frequency response curve** (courbe de réponse en fréquence): steady-state amplitude vs $\omega/\omega_0$.
4. What happens when $\zeta \to 0$ and $\omega = \omega_0$ (undamped resonance)? Solve and observe secular growth $\sim t\sin(\omega_0 t)$.

In [ ]:
# Forced oscillator — explore resonance
omega0 = 2.0
zeta = 0.1
F0 = 1.0

def forced_damped(t, state, omega):
    y, v = state
    return [v, F0*np.cos(omega*t) - 2*zeta*omega0*v - omega0**2*y]

fig, axes = plt.subplots(2, 1, figsize=(10, 8))
t_eval_res = np.linspace(0, 40, 1000)

# Time-domain solutions for different driving frequencies
for omega_drive in [1.0, 1.9, 2.0, 2.1, 3.0]:
    sol_res = solve_ivp(forced_damped, (0, 40), [0, 0],
                        args=(omega_drive,), t_eval=t_eval_res)
    axes[0].plot(sol_res.t, sol_res.y[0], label=f'$\\omega={omega_drive}$')

axes[0].set_xlabel('$t$'); axes[0].set_ylabel('$y(t)$')
axes[0].set_title('Forced oscillator at different driving frequencies')
axes[0].legend(fontsize=9); axes[0].grid(True, alpha=0.3)

# Frequency response curve (steady-state amplitude)
omega_ratio = np.linspace(0.1, 3, 500)
omega_values = omega_ratio * omega0
# Analytical steady-state amplitude:
# A = F0 / sqrt((omega0^2 - omega^2)^2 + (2*zeta*omega0*omega)^2)
amplitude = F0 / np.sqrt((omega0**2 - omega_values**2)**2 + (2*zeta*omega0*omega_values)**2)

for zeta_val in [0.05, 0.1, 0.3, 0.7]:
    amp = F0 / np.sqrt((omega0**2 - omega_values**2)**2 + (2*zeta_val*omega0*omega_values)**2)
    axes[1].plot(omega_ratio, amp, label=f'$\\zeta={zeta_val}$', linewidth=2)

axes[1].set_xlabel('$\\omega / \\omega_0$')
axes[1].set_ylabel('Steady-state amplitude')
axes[1].set_title('Frequency Response Curve (Courbe de réponse en fréquence)')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
# Figure: Top — time-domain response showing resonance amplification near omega = omega_0.
# Bottom — frequency response curves for different damping ratios.